In [2]:
import  pandas as pd
df = pd.DataFrame({'age':    [ 3,  29],
                   
                   'height': [94, 170],
                   'weight': [31, 115]})

In [37]:
val_splits = 0.2
test_splits = 0.1

from sklearn.model_selection import StratifiedGroupKFold
df = pd.read_csv("1517_merged_dataset_table.csv")
        #Create splitter that stratifies on 'genre' and groups by 'artist'
n_test_splits = int(1 / test_splits)
sgkf_test = StratifiedGroupKFold(n_splits=n_test_splits, shuffle=True, random_state=42)

# Get the first split: trainval vs test -> 80 / 20 split

for i , (trainval_idx, test_idx) in enumerate(sgkf_test.split(df, df['genre'], groups=df['artist'])): 
    if i == 0:                                         # sgkf makes sure that one artist can't end up in
        df_trainval = df.iloc[trainval_idx]                                           # multiple splits
        df_test = df.iloc[test_idx]
        break       # only take the first fold
# Get the second split: train vs val -> 80 / 20 split
n_val_splits = int(1 / val_splits)
sgkf_val = StratifiedGroupKFold(n_splits=n_val_splits, shuffle=True, random_state=42)  
for i, (train_idx, val_idx) in enumerate(sgkf_val.split(df_trainval, df_trainval['genre'], groups=df_trainval['artist'])):
    if i == 0:
        df_train = df_trainval.iloc[train_idx]
        df_val = df_trainval.iloc[val_idx]
        break       # only take the first fold
# check that splits indeed don't intersect

#artist_train =set(df_val["artist"])


In [ ]:
genres = set(df["genre"].unique())  # test_split 0
test_genre_counts = {}
for genre in genres:
    size = len(df_test[df_test["genre"] ==genre])
    test_genre_counts[genre] = size
dict(sorted(test_genre_counts.items(), key=lambda item: item[1]))
#

{'Reggae': 12,
 'Latin': 12,
 'Blues': 12,
 'Country': 13,
 'Classical': 14,
 'Jazz': 15,
 'Hip-Hop': 15,
 'New_Age': 19,
 'R_and_B_and_Soul': 20,
 'Rock_and_Pop': 21,
 'Electronic_and_Dance': 22,
 'Alternative_and_Punk': 24}

In [23]:
genres = set(df["genre"].unique())  # split 2
val_genre_counts = {}
for genre in genres:
    size = len(df_val[df_val["genre"] ==genre])
    val_genre_counts[genre] = size
dict(sorted(val_genre_counts.items(), key=lambda item: item[1]))
#

{'Latin': 18,
 'Alternative_and_Punk': 20,
 'Classical': 24,
 'Blues': 27,
 'Country': 28,
 'R_and_B_and_Soul': 29,
 'Hip-Hop': 32,
 'Electronic_and_Dance': 32,
 'Jazz': 33,
 'New_Age': 38,
 'Reggae': 39,
 'Rock_and_Pop': 40}

In [29]:
val_genre_counts.values()

dict_values([35, 27, 29, 21, 27, 29, 10, 20, 31, 35, 28, 30])

In [25]:
genres = set(df["genre"].unique())  # split 3
val_genre_counts = {}
for genre in genres:
    size = len(df_val[df_val["genre"] ==genre])
    val_genre_counts[genre] = size
dict(sorted(val_genre_counts.items(), key=lambda item: item[1]))

{'Classical': 14,
 'Latin': 19,
 'Electronic_and_Dance': 23,
 'Hip-Hop': 24,
 'R_and_B_and_Soul': 26,
 'New_Age': 29,
 'Reggae': 30,
 'Rock_and_Pop': 30,
 'Jazz': 31,
 'Alternative_and_Punk': 33,
 'Country': 46,
 'Blues': 48}

In [28]:
genres = set(df["genre"].unique())  # split 0
val_genre_counts = {}
for genre in genres:
    size = len(df_val[df_val["genre"] ==genre])
    val_genre_counts[genre] = size
dict(sorted(val_genre_counts.items(), key=lambda item: item[1]))

{'Classical': 10,
 'Electronic_and_Dance': 20,
 'Reggae': 21,
 'Jazz': 27,
 'R_and_B_and_Soul': 27,
 'Rock_and_Pop': 28,
 'New_Age': 29,
 'Hip-Hop': 29,
 'Blues': 30,
 'Latin': 31,
 'Country': 35,
 'Alternative_and_Punk': 35}

In [1]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd

def balanced_group_split(test_size, val_size,
                         group_col='artist',
                         class_col='genre',
                         tol=0.01,
                         seed=42,
                         max_tries=10000):
    """
    Returns train, val, test indices so that:
      - no artist overlaps splits
      - each split’s genre distribution is within tol of overall
    """
    rng = np.random.RandomState(seed)
    gss   = GroupShuffleSplit(n_splits=max_tries,
                              test_size=test_size,
                              random_state=seed)
    df = pd.read_csv(r"C:\Users\Kochana\projects\genres\1517_merged_dataset_table.csv")
    genres = df[class_col].value_counts(normalize=True)
    for trainval_idx, test_idx in gss.split(df, groups=df[group_col]):
        # check test balance
        test_dist = df.iloc[test_idx][class_col].value_counts(normalize=True)
        if np.max(np.abs(test_dist.reindex(genres.index, fill_value=0) - genres)) > tol:
            continue

        # now val split within trainval
        gss_val = GroupShuffleSplit(n_splits=max_tries,
                                    test_size=val_size/(1-test_size),
                                    random_state=seed)
                                                # multiple splits
        df_test = df.iloc[test_idx]
        df_trainval = df.iloc[trainval_idx]
        for train_idx, val_idx in gss_val.split(df_trainval, 
                                                groups=df_trainval[group_col]):
            val_dist = df_trainval.iloc[val_idx][class_col] \
                           .value_counts(normalize=True)
            if np.max(np.abs(val_dist.reindex(genres.index, fill_value=0) - genres)) <= tol:
                # success!
                # map back to original indices
                train_idx = trainval_idx[np.array(train_idx)]
                val_idx   = trainval_idx[np.array(val_idx)]
                df_train = df.iloc[train_idx]
                df_val= df.iloc[val_idx]
                return df_train, df_val, df_test

    raise RuntimeError("Could not find a balanced grouping within tol")


In [4]:
df_train, df_val, df_test = balanced_group_split(test_size= 0.1, val_size= 0.2,
                         group_col='artist',
                         class_col='genre',
                         tol=0.018,
                         seed=420000,
                         max_tries=10000)

In [5]:
df = pd.read_csv(r"C:\Users\Kochana\projects\genres\1517_merged_dataset_table.csv")
genres = set(df["genre"].unique())  # val split
test_genre_counts = {}
for genre in genres:
    size = len(df_val[df_val["genre"] ==genre])
    test_genre_counts[genre] = size
dict(sorted(test_genre_counts.items(), key=lambda item: item[1]))

{'Alternative_and_Punk': 28,
 'Pop': 29,
 'R_and_B_and_Soul': 29,
 'Jazz': 29,
 'Reggae': 31,
 'New_Age': 33,
 'Electronic_and_Dance': 33,
 'Blues': 34,
 'Latin': 35,
 'Country': 35,
 'Hip-Hop': 35,
 'Rock': 39,
 'Classical': 39}

In [3]:
df = pd.read_csv(r"C:\Users\Kochana\projects\genres\1517_merged_dataset_table.csv")
genres = set(df["genre"].unique())  # val split
test_genre_counts = {}
for genre in genres:
    size = len(df_val[df_val["genre"] ==genre])
    test_genre_counts[genre] = size
dict(sorted(test_genre_counts.items(), key=lambda item: item[1]))

{'Hip-Hop': 25,
 'Reggae': 27,
 'Classical': 27,
 'New_Age': 29,
 'R_and_B_and_Soul': 29,
 'Electronic_and_Dance': 31,
 'Blues': 33,
 'Country': 34,
 'Rock': 35,
 'Latin': 35,
 'Jazz': 35,
 'Pop': 36,
 'Alternative_and_Punk': 36}

In [14]:
df = pd.read_csv(r"C:\Polina\master\thesis\annotations\dataset\merged_dataset.csv")
genres = set(df["genre"].unique())  # test_split 0
test_genre_counts = {}
for genre in genres:
    size = len(df_test[df_test["genre"] ==genre])
    test_genre_counts[genre] = size
dict(sorted(test_genre_counts.items(), key=lambda item: item[1]))

{'Rock': 13,
 'Electronic_and_Dance': 14,
 'New_Age': 15,
 'Hip-Hop': 15,
 'Reggae': 16,
 'Blues': 16,
 'R_and_B_and_Soul': 17,
 'Classical': 17,
 'Pop': 17,
 'Alternative_and_Punk': 18,
 'Country': 18,
 'Folk': 20,
 'Jazz': 20,
 'Latin': 21}